# Ingestion_Knowledge (Colab)

Notebook de ingesta para persistir **todo** el conocimiento RAG en Google Drive usando **ChromaDB**.

Ruta persistente obligatoria:
`/content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base`

In [ ]:
# 1) Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Configuración de rutas persistentes
from pathlib import Path

PERSIST_BASE = Path('/content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base')
REPO_DIR = PERSIST_BASE / 'LLM-Juego-de-Tronos'
CHROMA_DIR = PERSIST_BASE / 'chroma_db'
EMBEDDING_DIR = PERSIST_BASE / 'embedding_model'
CHUNKS_DIR = PERSIST_BASE / 'chunks'

for p in [PERSIST_BASE, CHROMA_DIR, EMBEDDING_DIR, CHUNKS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Persist base:', PERSIST_BASE)
print('Repo dir    :', REPO_DIR)
print('Chroma dir  :', CHROMA_DIR)
print('Embed dir   :', EMBEDDING_DIR)
print('Chunks dir  :', CHUNKS_DIR)

In [ ]:
# 3) Clonar/actualizar repositorio dentro de Drive
# IMPORTANTE: cambia REPO_URL por tu remoto real si hace falta.
REPO_URL = 'https://github.com/<tu_usuario>/LLM-Juego-de-Tronos.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --rebase

%cd {REPO_DIR}
!git rev-parse --short HEAD

In [ ]:
# 4) Dependencias para ingesta + ChromaDB
!pip -q install --upgrade chromadb sentence-transformers transformers torch ebooklib beautifulsoup4 pandas pyarrow

In [ ]:
# 5) Ingesta EPUB -> chunks -> embeddings -> Chroma persistente
import re
import uuid
import pandas as pd
from bs4 import BeautifulSoup
import ebooklib
from ebooklib import epub
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

EPUB_CANDIDATES = [
    REPO_DIR / 'Canci_243_n_de_Hielo_y_Fuego_01_-_Juego_de_Tronos.epub',
    REPO_DIR / 'juego_de_tronos.epub',
]

EPUB_PATH = None
for c in EPUB_CANDIDATES:
    if c.exists():
        EPUB_PATH = c
        break

if EPUB_PATH is None:
    raise FileNotFoundError(f'No encuentro EPUB en {EPUB_CANDIDATES}')

print('Usando EPUB:', EPUB_PATH)

MODEL_NAME = 'Alibaba-NLP/gte-large-en-v1.5'
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200
BATCH_SIZE = 64
COLLECTION_NAME = 'got_book1_chunks'

def clean_text(text: str) -> str:
    text = re.sub(r'\\s+', ' ', text)
    return text.strip()

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if chunk:
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break
    return chunks

def load_epub_sections(epub_path):
    book = epub.read_epub(str(epub_path))
    sections = []
    chapter_idx = 0

    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        html = item.get_content()
        soup = BeautifulSoup(html, 'html.parser')
        text = clean_text(soup.get_text(' ', strip=True))
        if len(text) < 120:
            continue
        chapter_idx += 1
        sections.append({
            'chapter_index': chapter_idx,
            'chapter_id': item.get_id(),
            'chapter_name': item.get_name(),
            'text': text,
        })

    return sections

sections = load_epub_sections(EPUB_PATH)
print('Capítulos válidos:', len(sections))

records = []
for sec in sections:
    c_chunks = chunk_text(sec['text'])
    for j, ch in enumerate(c_chunks):
        records.append({
            'chunk_id': f"{sec['chapter_id']}_{j}",
            'chapter_index': sec['chapter_index'],
            'chapter_id': sec['chapter_id'],
            'chapter_name': sec['chapter_name'],
            'chunk_index': j,
            'text': ch,
            'char_len': len(ch),
        })

chunks_df = pd.DataFrame(records)
print('Chunks totales:', len(chunks_df))

chunks_parquet = CHUNKS_DIR / 'chunks.parquet'
chunks_df.to_parquet(chunks_parquet, index=False)
print('Parquet guardado en:', chunks_parquet)

embedder = SentenceTransformer(MODEL_NAME)
embedder.save(str(EMBEDDING_DIR))
print('Modelo guardado en:', EMBEDDING_DIR)

client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(anonymized_telemetry=False))
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(name=COLLECTION_NAME, metadata={'source': 'game_of_thrones_book_1'})

texts = chunks_df['text'].tolist()
metadatas = chunks_df[['chapter_index', 'chapter_id', 'chapter_name', 'chunk_index', 'char_len']].to_dict(orient='records')
ids = [str(uuid.uuid4()) for _ in texts]

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_meta = metadatas[i:i+BATCH_SIZE]
    batch_ids = ids[i:i+BATCH_SIZE]
    batch_embs = embedder.encode(batch_texts, normalize_embeddings=True, show_progress_bar=False).tolist()

    collection.add(ids=batch_ids, embeddings=batch_embs, documents=batch_texts, metadatas=batch_meta)

print('Colección Chroma creada:', COLLECTION_NAME)
print('Total en colección:', collection.count())
print('DB persistida en:', CHROMA_DIR)

In [ ]:
# 6) Smoke test de recuperación
query = '¿Qué sabemos sobre la casa Stark en Invernalia?'
q_emb = embedder.encode([query], normalize_embeddings=True).tolist()[0]
res = collection.query(query_embeddings=[q_emb], n_results=5)

for i, doc in enumerate(res['documents'][0], start=1):
    meta = res['metadatas'][0][i-1]
    print(f"[{i}] cap={meta['chapter_index']} chunk={meta['chunk_index']} name={meta['chapter_name']}")
    print(doc[:300], '...')
    print('-' * 80)